# 01 — Data Understanding

**Goal:** load the raw Telco Customer Churn dataset and understand exactly what we have
*before* touching it: structure, types, quality problems, and the meaning of every column.

Nothing is modified in this notebook. Every cleaning decision made later (notebook 02)
is justified by evidence gathered here.

**Dataset:** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn),
originally an IBM sample dataset. One row = one customer snapshot; target = `Churn`.


In [1]:
import sys
from pathlib import Path

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import RAW_DATA_FILE, TARGET_COLUMN, ID_COLUMN

pd.set_option("display.max_columns", None)
print(f"pandas {pd.__version__}")

pandas 3.0.5


## 1. Load the raw data

We load with **no type coercion or cleaning** — we want to see the file as it truly is.

In [2]:
df = pd.read_csv(RAW_DATA_FILE)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 7,043  |  Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Structure: data types and non-null counts

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**Observations:**

- Only 3 columns load as numeric: `SeniorCitizen` (int64), `tenure` (int64), `MonthlyCharges` (float64).
- `TotalCharges` loads as a **string**, even though it should be a dollar amount. That is a red flag
  we investigate in §6 — something non-numeric must be hiding in it.
- `SeniorCitizen` loads as an integer, but it is conceptually a Yes/No flag like `Partner` and
  `Dependents` — the dataset just happens to encode this one as 0/1 (§7).
- Pandas reports zero nulls — but as `TotalCharges` shows, *"no NaN" is not the same as "no missing
  data"*. Missingness can hide inside string columns.


## 3. Duplicates and identifier integrity

If `customerID` is not unique, rows are not independent customers and everything downstream changes.

In [4]:
print(f"Exact duplicate rows:      {df.duplicated().sum()}")
print(f"Duplicated customerIDs:    {df[ID_COLUMN].duplicated().sum()}")
print(f"Unique customerIDs:        {df[ID_COLUMN].nunique():,} of {len(df):,} rows")

Exact duplicate rows:      0
Duplicated customerIDs:    0
Unique customerIDs:        7,043 of 7,043 rows


**Finding:** no duplicate rows, and `customerID` is unique — one row per customer, as expected.
`customerID` is a random identifier with no predictive meaning; it will be **excluded from modeling**
(keeping it would at best do nothing and at worst let a tree model memorize customers).
We keep it as an index for reporting and scoring output.

## 4. Target variable: `Churn`

In [5]:
counts = df[TARGET_COLUMN].value_counts()
pct = df[TARGET_COLUMN].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({"customers": counts, "percent": pct})

,customers,percent
Churn,,
No,5174,73.46
Yes,1869,26.54


**Finding:** 1,869 of 7,043 customers churned — a **26.54% churn rate**.

Implications we carry through the whole project:

1. **Class imbalance** (roughly 1:2.8). Not extreme, but enough that *accuracy is misleading*:
   a useless model that predicts "No churn" for everyone scores ~73.5% accuracy. We will use
   stratified splits, judge models on recall/ROC-AUC/PR-AUC, and evaluate imbalance handling
   explicitly (class weights vs. threshold tuning vs. resampling).
2. **The minority class is the one the business cares about.** Missing a churner (false negative)
   means losing a customer we could have tried to retain.


## 5. Categorical columns: cardinality and levels

Low, clean cardinality is good news for one-hot encoding. We also check for messy variants of the same category (casing, whitespace).

In [6]:
cat_cols = [c for c in df.columns
            if c not in (ID_COLUMN, "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges")]
for c in cat_cols:
    print(f"{c:18s} ({df[c].nunique()}): {sorted(df[c].unique())}")

gender             (2): ['Female', 'Male']
Partner            (2): ['No', 'Yes']
Dependents         (2): ['No', 'Yes']
PhoneService       (2): ['No', 'Yes']
MultipleLines      (3): ['No', 'No phone service', 'Yes']
InternetService    (3): ['DSL', 'Fiber optic', 'No']
OnlineSecurity     (3): ['No', 'No internet service', 'Yes']
OnlineBackup       (3): ['No', 'No internet service', 'Yes']
DeviceProtection   (3): ['No', 'No internet service', 'Yes']
TechSupport        (3): ['No', 'No internet service', 'Yes']
StreamingTV        (3): ['No', 'No internet service', 'Yes']
StreamingMovies    (3): ['No', 'No internet service', 'Yes']
Contract           (3): ['Month-to-month', 'One year', 'Two year']
PaperlessBilling   (2): ['No', 'Yes']
PaymentMethod      (4): ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn              (2): ['No', 'Yes']


**Observations:**

- All categoricals are clean and low-cardinality (2–4 levels). No casing or spelling variants —
  one-hot encoding will be safe and compact.
- Six internet add-on columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
  `StreamingTV`, `StreamingMovies`) share a third level **"No internet service"**, and
  `MultipleLines` has **"No phone service"**. These are *structural* values: they repeat, in a
  different column, information already carried by `InternetService`/`PhoneService`.
  Whether to keep them as distinct levels or collapse them to "No" is a real modeling decision —
  we defer it to feature engineering (notebook 05) and will justify it there.


In [7]:
# The "No internet service" level appears exactly as often as InternetService == "No" — confirming
# it is structural, not independent information.
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
no_internet = (df["InternetService"] == "No").sum()
print(f'InternetService == "No": {no_internet}')
for c in addon_cols:
    n = (df[c] == "No internet service").sum()
    assert n == no_internet
print("All six add-on columns have exactly", no_internet, '"No internet service" rows — structural, as expected.')

InternetService == "No": 1526
All six add-on columns have exactly 1526 "No internet service" rows — structural, as expected.


## 6. The `TotalCharges` problem

`TotalCharges` should be numeric but loaded as a string. Let's find out why.

In [8]:
tc_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = df[tc_numeric.isna()]
print(f"Rows where TotalCharges is not parseable as a number: {len(bad)}")
print(f"Raw value in those rows: {[repr(v) for v in bad['TotalCharges'].unique()]}")
bad[[ID_COLUMN, "tenure", "Contract", "MonthlyCharges", "TotalCharges", TARGET_COLUMN]]

Rows where TotalCharges is not parseable as a number: 11
Raw value in those rows: ["' '"]


,customerID,tenure,Contract,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,Two year,52.55,,No
753,3115-CZMZD,0,Two year,20.25,,No
936,5709-LVOEQ,0,Two year,80.85,,No
1082,4367-NUYAO,0,Two year,25.75,,No
1340,1371-DWPAZ,0,Two year,56.05,,No
3331,7644-OMVMY,0,Two year,19.85,,No
3826,3213-VVOLG,0,Two year,25.35,,No
4380,2520-SGTTA,0,Two year,20.00,,No
5218,2923-ARZLG,0,One year,19.70,,No
6670,4075-WKNIU,0,Two year,73.35,,No


**Finding — the dataset's one genuine data-quality issue:**

- **11 rows** have `TotalCharges` equal to a single **space character** `" "` — hidden missingness
  that `isna()` cannot see.
- All 11 have **`tenure` = 0**: they are brand-new customers who have not yet been billed.
  The blank is therefore *structurally meaningful* (nothing billed yet), not a recording error.
- All 11 are on one- or two-year contracts and none has churned.

This drives a cleaning decision in notebook 02, where we will weigh the two defensible options —
drop the 11 rows (0.16% of data) vs. impute `TotalCharges = 0` — and justify the choice.


In [9]:
# Sanity check: is TotalCharges ≈ tenure × MonthlyCharges? If roughly yes, the column is
# consistent and largely derived — relevant later for both feature engineering and leakage thinking.
mask = tc_numeric.notna() & (df["tenure"] > 0)
approx = df.loc[mask, "tenure"] * df.loc[mask, "MonthlyCharges"]
rel_diff = ((tc_numeric[mask] - approx).abs() / tc_numeric[mask])
print(f"Median |TotalCharges − tenure×MonthlyCharges| / TotalCharges: {rel_diff.median():.1%}")

Median |TotalCharges − tenure×MonthlyCharges| / TotalCharges: 2.0%


`TotalCharges` tracks `tenure × MonthlyCharges` within ~2% (median) — the small gap is consistent
with customers' monthly price changing over their lifetime. So `TotalCharges` is nearly collinear
with `tenure` and `MonthlyCharges` combined. It is **not** target leakage (it's known at prediction
time), but the near-redundancy matters for interpreting linear-model coefficients later.

## 7. `SeniorCitizen`: an integer that is really a category

In [10]:
df["SeniorCitizen"].value_counts()

SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

**Finding:** `SeniorCitizen` is 0/1 (16.2% seniors) while every other demographic flag is Yes/No.
Left as an integer it would work mechanically, but for consistency and readable
outputs (plots, SHAP values, app inputs) we will treat it as a **categorical** and map it to
No/Yes during cleaning. This is a representation choice, not a correction — both encodings carry
identical information for the models we'll use.

## 8. Numeric summary statistics

In [11]:
summary = df[["tenure", "MonthlyCharges"]].assign(TotalCharges=tc_numeric).describe().round(2)
summary

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7032.00
mean,32.37,64.76,2283.30
std,24.56,30.09,2266.77
min,0.00,18.25,18.80
25%,9.00,35.50,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.85,3794.74
max,72.00,118.75,8684.80


**Observations:**

- `tenure`: 0–72 months, mean ≈ 32.4, median 29 — the panel spans six years, with mass at both
  extremes (we'll see a U-shaped distribution in EDA: many new customers, many long-tenured ones).
- `MonthlyCharges`: \$18.25–\$118.75, mean ≈ \$64.76.
- `TotalCharges`: count is **7,032** (= 7,043 − 11 blanks), max ≈ \$8,684.80, and its scale/skew
  reflects its `tenure × MonthlyCharges` structure. No negative or impossible values anywhere.


## 9. Data dictionary

| Variable | Represents | Raw type | Treat as | Use in model? | Notes / problems |
|---|---|---|---|---|---|
| `customerID` | Unique customer identifier | str | identifier | **No** | Random ID; keep only for reporting/scoring output |
| `gender` | Male/Female | str | categorical | Yes | Clean binary |
| `SeniorCitizen` | Senior citizen flag | int 0/1 | **categorical** | Yes | Only demographic coded 0/1; map to No/Yes for consistency |
| `Partner` | Has a partner | str | categorical | Yes | Clean binary |
| `Dependents` | Has dependents | str | categorical | Yes | Clean binary |
| `tenure` | Months with the company | int | numerical | Yes | 0–72; eleven 0-tenure rows tie to the TotalCharges blanks |
| `PhoneService` | Has phone service | str | categorical | Yes | Clean binary |
| `MultipleLines` | Multiple phone lines | str | categorical | Yes | 3rd level "No phone service" is structural (≡ PhoneService=No) |
| `InternetService` | DSL / Fiber optic / No | str | categorical | Yes | Key service variable |
| `OnlineSecurity` | Add-on: online security | str | categorical | Yes | 3rd level "No internet service" structural |
| `OnlineBackup` | Add-on: online backup | str | categorical | Yes | Same structural level |
| `DeviceProtection` | Add-on: device protection | str | categorical | Yes | Same structural level |
| `TechSupport` | Add-on: tech support | str | categorical | Yes | Same structural level |
| `StreamingTV` | Add-on: streaming TV | str | categorical | Yes | Same structural level |
| `StreamingMovies` | Add-on: streaming movies | str | categorical | Yes | Same structural level |
| `Contract` | Month-to-month / One year / Two year | str | categorical | Yes | Expected strong churn driver; ordinal in commitment but we one-hot it |
| `PaperlessBilling` | Paperless billing | str | categorical | Yes | Clean binary |
| `PaymentMethod` | 4 payment methods | str | categorical | Yes | Two "(automatic)" methods vs two manual — worth a derived feature |
| `MonthlyCharges` | Current monthly bill (\$) | float | numerical | Yes | Clean |
| `TotalCharges` | Lifetime amount billed (\$) | **str** | numerical | Yes (with care) | 11 blank strings (tenure=0); nearly ≡ tenure×MonthlyCharges → collinearity |
| `Churn` | Left within the last month | str | **target** | — | Yes/No → 1/0; 26.54% positive |


## 10. Summary of data-quality issues → cleaning plan

| # | Issue | Evidence | Planned handling (notebook 02) |
|---|---|---|---|
| 1 | `TotalCharges` stored as string | dtype `str` on load | Convert to numeric |
| 2 | 11 hidden missing values (`" "`) in `TotalCharges` | §6 | Decide: drop vs. impute 0 — resolved with justification in 02 |
| 3 | `SeniorCitizen` encoded 0/1 unlike sibling columns | §7 | Map to No/Yes, treat as categorical |
| 4 | Structural "No internet/phone service" levels | §5 | Defer to feature engineering; decide collapse vs. keep |
| 5 | Target is Yes/No strings | §4 | Encode 1/0 at modeling time |
| 6 | Class imbalance 26.5/73.5 | §4 | Stratify splits; imbalance-aware metrics & strategies |

**Not issues:** no NaN, no duplicates, no impossible values, no messy category labels.
This is a clean dataset by industry standards — its difficulty is analytical, not janitorial.
